# CosyVoice 2 — Indian English TTS

**Apache 2.0 license** — enterprise/commercial use OK.

Run all cells top to bottom. Tested on RunPod RTX 4090 and Colab A100.

| Step | What | Time |
|------|------|------|
| 1 | Install everything | ~10 min |
| 2 | Download model + data | ~10 min |
| 3 | Load model | ~1 min |
| 4 | Browse & pick reference voices | ~5 min |
| 5 | Generate podcast | ~10 min |
| 6 | Listen & save | ~1 min |

## Step 1: Install Everything

In [ ]:
import os, sys, subprocess

# Detect environment
IS_COLAB = os.path.exists('/content') and 'COLAB_RELEASE_TAG' in os.environ
BASE = '/content' if IS_COLAB else '/workspace'
print(f"Environment: {'Colab' if IS_COLAB else 'RunPod/Other'}")
print(f"Base: {BASE}")

# GPU check
import torch
assert torch.cuda.is_available(), "No GPU!"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
# System deps
!apt-get -qq update > /dev/null 2>&1
!apt-get -qq install -y sox libsox-dev ffmpeg > /dev/null 2>&1

# Clone CosyVoice
os.chdir(BASE)
COSYVOICE_DIR = f'{BASE}/CosyVoice'
if not os.path.exists(f'{COSYVOICE_DIR}/.git'):
    !rm -rf {COSYVOICE_DIR}
    !git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git {COSYVOICE_DIR}
    !cd {COSYVOICE_DIR} && git submodule update --init --recursive
print("Repo cloned.")

# Install CosyVoice deps — skip torch/torchaudio (already installed)
# Install line by line to avoid one bad package killing everything
!cd {COSYVOICE_DIR} && cat requirements.txt | grep -v '^torch' | grep -v tensorrt | grep -v ttsfrd | while read pkg; do pip install -q "$pkg" 2>/dev/null; done

# Ensure critical packages are installed
!pip install -q hyperpyyaml modelscope onnxruntime soundfile librosa \
    openai-whisper conformer diffsptk inflect pydub einops omegaconf \
    huggingface_hub datasets torchaudio num2words 'transformers>=4.45,<4.50' \
    2>&1 | tail -3

print("\nInstall complete!")

## Step 2: HuggingFace Login + Download Model + Data

In [ ]:
# --- HuggingFace Login ---
try:
    if IS_COLAB:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

if not os.environ.get('HF_TOKEN'):
    print("HF_TOKEN not set.")
    print("Get a token at: https://huggingface.co/settings/tokens")
    print("Make sure 'Access to public gated repos' is enabled.")
    from huggingface_hub import login
    login()
else:
    from huggingface_hub import HfApi
    try:
        print(f"HF user: {HfApi().whoami()['name']}")
    except Exception:
        from huggingface_hub import login
        login()

In [ ]:
# --- Download CosyVoice2 Model ---
sys.path.insert(0, COSYVOICE_DIR)
sys.path.insert(0, f'{COSYVOICE_DIR}/third_party/Matcha-TTS')

MODEL_DIR = f'{COSYVOICE_DIR}/pretrained_models/CosyVoice2-0.5B'
if not os.path.exists(f'{MODEL_DIR}/llm.pt'):
    from huggingface_hub import snapshot_download
    snapshot_download('FunAudioLLM/CosyVoice2-0.5B', local_dir=MODEL_DIR)
    print("Model downloaded!")
else:
    print("Model already downloaded.")

In [ ]:
# --- Download Svarah Indian English Reference Voices ---
import io, random
import numpy as np
import soundfile as sf

DATA_DIR = f'{BASE}/data'
BACKUP_DIR = f'{BASE}/backup_cosyvoice2'
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BACKUP_DIR = '/content/drive/MyDrive/indian_tts_cosyvoice2'
os.makedirs(BACKUP_DIR, exist_ok=True)
os.makedirs(f'{DATA_DIR}/svarah/male', exist_ok=True)
os.makedirs(f'{DATA_DIR}/svarah/female', exist_ok=True)

import glob
existing_male = glob.glob(f'{DATA_DIR}/svarah/male/svarah_*.wav')
existing_female = glob.glob(f'{DATA_DIR}/svarah/female/svarah_*.wav')

if len(existing_male) >= 5 and len(existing_female) >= 5:
    print(f"Svarah data exists: {len(existing_male)} male, {len(existing_female)} female clips")
else:
    print("Downloading Svarah reference voices...")
    from datasets import load_dataset, Audio

    ds = load_dataset("ai4bharat/Svarah", split="test")
    ds = ds.cast_column("audio_filepath", Audio(decode=False))

    male_saved = 0
    female_saved = 0
    manifests = []

    for i, sample in enumerate(ds):
        if male_saved >= 20 and female_saved >= 20:
            break

        gender = (sample.get('gender') or '').strip().lower()
        text = (sample.get('text') or '').strip()
        if gender not in ('male', 'female') or len(text) < 3:
            continue
        if gender == 'male' and male_saved >= 20:
            continue
        if gender == 'female' and female_saved >= 20:
            continue

        audio_data = sample.get('audio_filepath')
        if not audio_data or not audio_data.get('bytes'):
            continue

        try:
            arr, sr = sf.read(io.BytesIO(audio_data['bytes']))
            arr = arr.astype(np.float32)
            if arr.ndim > 1:
                arr = arr.mean(axis=1)
        except Exception:
            continue

        duration = len(arr) / sr
        if duration < 3.0 or duration > 15.0:
            continue

        mx = np.abs(arr).max()
        if mx > 0:
            arr = arr / mx * 0.95

        filepath = f'{DATA_DIR}/svarah/{gender}/svarah_{i:06d}.wav'
        sf.write(filepath, arr, sr)
        manifests.append(f"{filepath}|{0 if gender=='male' else 1}|{text}")

        if gender == 'male':
            male_saved += 1
        else:
            female_saved += 1

    # Write manifest
    with open(f'{DATA_DIR}/train.txt', 'w') as f:
        f.write('# audio_path|speaker_id|text\n')
        for line in manifests:
            f.write(line + '\n')

    print(f"Done! Male: {male_saved} | Female: {female_saved} clips")

print("\nStep 2 complete!")

## Step 3: Load CosyVoice 2 Model

In [ ]:
os.chdir(COSYVOICE_DIR)
from cosyvoice.cli.cosyvoice import CosyVoice2
import torchaudio

cosyvoice = CosyVoice2('pretrained_models/CosyVoice2-0.5B')
print(f"Model loaded! Sample rate: {cosyvoice.sample_rate}")

## Step 4: Browse & Pick Reference Voices

Listen to several clips and pick the ones with the accent you want.
- For **lighter Indian accent** (corporate English): pick speakers who sound less regional
- For **stronger Indian accent**: pick speakers with more regional flavor

The model clones whatever accent it hears in the reference.

In [ ]:
import IPython.display as ipd

male_wavs = sorted(glob.glob(f'{DATA_DIR}/svarah/male/svarah_*.wav'))
female_wavs = sorted(glob.glob(f'{DATA_DIR}/svarah/female/svarah_*.wav'))

# Read manifest for transcripts
transcripts = {}
manifest_path = f'{DATA_DIR}/train.txt'
if os.path.exists(manifest_path):
    with open(manifest_path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split('|')
            transcripts[parts[0]] = parts[2]

print("=== MALE SPEAKERS — pick the best accent ===")
for i, wav in enumerate(male_wavs[:10]):
    data, sr = sf.read(wav)
    dur = len(data) / sr
    txt = transcripts.get(wav, '(no transcript)')
    print(f"\nMale #{i} ({dur:.1f}s): {txt[:70]}...")
    ipd.display(ipd.Audio(wav))

print("\n\n=== FEMALE SPEAKERS — pick the best accent ===")
for i, wav in enumerate(female_wavs[:10]):
    data, sr = sf.read(wav)
    dur = len(data) / sr
    txt = transcripts.get(wav, '(no transcript)')
    print(f"\nFemale #{i} ({dur:.1f}s): {txt[:70]}...")
    ipd.display(ipd.Audio(wav))

In [ ]:
# === SET YOUR CHOSEN REFERENCE CLIPS HERE ===
# Change the index numbers based on what sounded best above
MALE_IDX = 0      # Change this to the male clip # you liked
FEMALE_IDX = 0     # Change this to the female clip # you liked

male_ref = male_wavs[MALE_IDX]
female_ref = female_wavs[FEMALE_IDX]
male_ref_text = transcripts.get(male_ref, '')
female_ref_text = transcripts.get(female_ref, '')

print(f"Male ref: #{MALE_IDX} — {male_ref_text[:60]}...")
print(f"Female ref: #{FEMALE_IDX} — {female_ref_text[:60]}...")

## Step 5: Generate Production Podcast

Features:
- **Per-line speed control** — higher speed = more energy
- **Phonetic pronunciation fixes** — "Preeya" not "Priya", "A I" not "AI"
- **Natural fillers** — "hmm", "right", "absolutely" between speaker switches
- **Smart pauses** — longer after questions, natural between hosts
- **Backup to Drive/local** automatically

In [ ]:
import time

# ============================================================
# PODCAST SCRIPT — (speaker, text, speed)
# Speed: 1.0=calm, 1.1=normal energy, 1.2=energetic, 1.3=very energetic
# Pronunciation: use phonetic spelling for problem words
#   AI -> "ay eye" or "A I"     Priya -> "Preeya"
#   startup -> "start up"       healthcare -> "health care"
#   dataset -> "data set"
# ============================================================

PODCAST_SCRIPT = [
    ("female", "Welcome to A I India, the podcast where we explore how artificial intelligence is transforming our country. I am Preeya.", 1.25),
    ("male", "And I am Arjun. Today we are talking about something really exciting. The rise of Indian A I start ups.", 1.25),
    ("female", "That is right, Arjun. India now has over three hundred A I start ups, and that number is growing every single month.", 1.2),
    ("male", "What I find really interesting is that many of these companies are solving uniquely Indian problems. Like agriculture, health care in rural areas, and education.", 1.15),
    ("female", "Absolutely. Take for example an A I system that can detect crop diseases just by looking at a photo taken on a farmer's mobile phone.", 1.2),
    ("male", "And in health care, A I models are now screening for conditions like diabetic retinopathy and tuberculosis in areas where there are very few doctors available.", 1.15),
    ("female", "The language barrier is another big challenge that A I is helping with. India has twenty two official languages and hundreds of dialects.", 1.15),
    ("male", "Exactly. And that is precisely why building speech technology like text to speech systems in Indian languages is so important.", 1.2),
    ("female", "Speaking of which, the progress in Indian language A I has been remarkable. Models can now understand and generate speech in Hindi, Tamil, Bengali, and many more.", 1.15),
    ("male", "The government has also been supportive with initiatives to build open source data sets for Indian languages. This is a game changer.", 1.2),
    ("female", "So what do you think is next for A I in India, Arjun?", 1.25),
    ("male", "I believe we will see A I becoming a part of everyday life. From voice assistants that truly understand Indian accents, to A I tutors that teach children in their mother tongue.", 1.15),
    ("female", "That is a beautiful vision. And it all starts with building the right foundation, the right data, the right models, and the right talent.", 1.1),
    ("male", "Could not agree more. India has the talent, and now we are building the tools.", 1.25),
    ("female", "That is all for today's episode of A I India. Thank you for listening, and we will see you next week.", 1.2),
    ("male", "Goodbye everyone, and keep innovating!", 1.3),
]

# ============================================================
# PAUSE SETTINGS (seconds) — tune these for conversation feel
# ============================================================
PAUSE_SAME_SPEAKER = 0.3       # Between lines from same speaker
PAUSE_SWITCH_SPEAKER = 0.5     # When switching speakers
PAUSE_AFTER_QUESTION = 0.8     # After a question mark
PAUSE_AFTER_FILLER = 0.4       # After filler word, before main statement

# ============================================================
# FILLER WORDS — acknowledgements between speaker switches
# ============================================================
FILLERS_MALE = ["hmm", "right", "yes", "absolutely"]
FILLERS_FEMALE = ["hmm", "yes", "right", "indeed"]
FILLER_SPEED = 1.0              # Normal speed so fillers are clearly heard
ADD_FILLERS = True              # Set False to disable

def generate_filler(cosyvoice_model, word, ref_wav, ref_txt, sample_rate):
    """Generate a filler word — spoken clearly, not rushed."""
    try:
        chunks = []
        for result in cosyvoice_model.inference_zero_shot(
            word + ".", ref_txt, ref_wav, stream=False, speed=FILLER_SPEED
        ):
            chunks.append(result['tts_speech'].squeeze().numpy())
        if chunks:
            audio = np.concatenate(chunks)
            max_len = int(sample_rate * 1.2)
            audio = audio[:max_len]
            # Trim trailing silence
            threshold = 0.01
            for end_idx in range(len(audio) - 1, 0, -1):
                if abs(audio[end_idx]) > threshold:
                    break
            audio = audio[:end_idx + int(sample_rate * 0.05)]
            fade = min(int(sample_rate * 0.05), len(audio))
            audio[-fade:] *= np.linspace(1, 0, fade)
            return audio * 0.7
    except Exception:
        pass
    return np.zeros(int(sample_rate * 0.3))

# ============================================================
# GENERATE PODCAST
# ============================================================
OUTPUT_DIR = f'{BASE}/outputs/cosyvoice2_podcast'
os.makedirs(OUTPUT_DIR, exist_ok=True)
sr = cosyvoice.sample_rate

all_segments = []
prev_speaker = None
filler_idx_male = 0
filler_idx_female = 0

print("Generating production podcast...\n")
start = time.time()

for i, (speaker, text, speed) in enumerate(PODCAST_SCRIPT):
    name = "Priya" if speaker == "female" else "Arjun"
    ref_wav = female_ref if speaker == "female" else male_ref
    ref_txt = female_ref_text if speaker == "female" else male_ref_text

    # --- Smart pauses and fillers ---
    if prev_speaker is not None:
        prev_text = PODCAST_SCRIPT[i-1][1]

        if prev_text.endswith('?'):
            all_segments.append(np.zeros(int(sr * PAUSE_AFTER_QUESTION)))
        elif speaker != prev_speaker:
            all_segments.append(np.zeros(int(sr * PAUSE_SWITCH_SPEAKER)))

            # Add filler from the NEW speaker (skip for intro/outro lines)
            if ADD_FILLERS and i > 1 \
               and not text.startswith("Welcome") \
               and not text.startswith("That is all") \
               and not text.startswith("Goodbye"):
                if speaker == "male":
                    fw = FILLERS_MALE[filler_idx_male % len(FILLERS_MALE)]
                    filler_idx_male += 1
                else:
                    fw = FILLERS_FEMALE[filler_idx_female % len(FILLERS_FEMALE)]
                    filler_idx_female += 1

                print(f"          ({name}: \"{fw}\")")
                filler = generate_filler(cosyvoice, fw, ref_wav, ref_txt, sr)
                all_segments.append(filler)
                all_segments.append(np.zeros(int(sr * PAUSE_AFTER_FILLER)))
        else:
            all_segments.append(np.zeros(int(sr * PAUSE_SAME_SPEAKER)))

    # --- Generate main line ---
    gen_start = time.time()
    chunks = []
    for result in cosyvoice.inference_zero_shot(text, ref_txt, ref_wav, stream=False, speed=speed):
        chunks.append(result['tts_speech'].squeeze().numpy())

    audio = np.concatenate(chunks) if chunks else np.zeros(sr)

    # Gentle fade in/out
    fade_len = min(int(sr * 0.02), len(audio) // 4)
    audio[:fade_len] *= np.linspace(0, 1, fade_len)
    audio[-fade_len:] *= np.linspace(1, 0, fade_len)

    gen_time = time.time() - gen_start
    duration = len(audio) / sr
    print(f"  [{name:5s}] {duration:.1f}s (spd:{speed:.2f}) | {text[:50]}...")

    sf.write(f'{OUTPUT_DIR}/line_{i:02d}_{speaker}.wav', audio, sr)
    all_segments.append(audio)
    prev_speaker = speaker

# Combine
full_audio = np.concatenate(all_segments)
PODCAST_PATH = f'{OUTPUT_DIR}/podcast_full.wav'
sf.write(PODCAST_PATH, full_audio, sr)

total_time = time.time() - start
total_dur = len(full_audio) / sr
print(f"\nPodcast done!")
print(f"  Duration: {total_dur:.0f}s ({total_dur/60:.1f} min)")
print(f"  Generation time: {total_time:.0f}s")

# Auto-backup
import shutil
shutil.copy2(PODCAST_PATH, os.path.join(BACKUP_DIR, 'podcast_production.wav'))
for f in glob.glob(f'{OUTPUT_DIR}/line_*.wav'):
    shutil.copy2(f, BACKUP_DIR)
print(f"  Backed up to: {BACKUP_DIR}")

## Step 6: Listen & Save

In [ ]:
print("=" * 60)
print("  CosyVoice 2 — Production Podcast")
print("  Indian English | Per-line energy | Fillers | Smart pauses")
print("=" * 60)

print("\nFull podcast:")
ipd.display(ipd.Audio(PODCAST_PATH))

print("\nAll lines:")
for i, (speaker, text, speed) in enumerate(PODCAST_SCRIPT):
    name = "Priya" if speaker == "female" else "Arjun"
    print(f"\n  [{name}] (speed:{speed}) {text[:55]}...")
    ipd.display(ipd.Audio(f'{OUTPUT_DIR}/line_{i:02d}_{speaker}.wav'))

In [ ]:
# Save to backup
import shutil
shutil.copy2(PODCAST_PATH, os.path.join(BACKUP_DIR, 'podcast_zero_shot.wav'))
for f in glob.glob(f'{OUTPUT_DIR}/line_*.wav'):
    shutil.copy2(f, BACKUP_DIR)
print(f"All files saved to: {BACKUP_DIR}")
print(f"\nDownload podcast_full.wav from: {PODCAST_PATH}")

## Pronunciation Testing

If any word sounds wrong, test phonetic spellings here. Common fixes:
- `AI` -> `A I` or `ay eye`
- `Priya` -> `Preeya`
- `startup` -> `start up`
- `healthcare` -> `health care`
- `dataset` -> `data set`

In [ ]:
# Test pronunciation of specific words/phrases
# Change the text below and run to hear how it sounds

test_text = "Welcome to A I India. I am Preeya."
test_speed = 1.2  # Adjust energy

print("[MALE]")
for result in cosyvoice.inference_zero_shot(test_text, male_ref_text, male_ref, stream=False, speed=test_speed):
    test_audio = result['tts_speech'].squeeze().numpy()
ipd.display(ipd.Audio(test_audio, rate=sr))

print("[FEMALE]")
for result in cosyvoice.inference_zero_shot(test_text, female_ref_text, female_ref, stream=False, speed=test_speed):
    test_audio = result['tts_speech'].squeeze().numpy()
ipd.display(ipd.Audio(test_audio, rate=sr))